<a href="https://colab.research.google.com/github/thiennguyen37-qn/Sofascore-stats/blob/main/sofascore_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cloudscraper
!pip install curl_cffi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 8.2 MB/s eta 0:00:00


In [ ]:
from curl_cffi.curl import debug_function
from curl_cffi import requests
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def scrape_sofascore_data(tournament_id, season_id):
    BASE_URL = f"https://www.sofascore.com/api/v1/unique-tournament/{tournament_id}/season/{season_id}/statistics"
    HEADERS = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Referer": "https://www.sofascore.com/"
    }
    FIELD_GROUPS = [
        "totalShots,goals,expectedGoals,goalConversionPercentage,leftFootGoals,rightFootGoals",
        "headedGoals,bigChancesMissed,successfulDribbles,successfulDribblesPercentage,goalsFromInsideTheBox,goalsFromOutsideTheBox",
        "tackles,interceptions,clearances,cleanSheet,outfielderBlocks",
        "penaltyConceded,errorLeadToGoal,errorLeadToShot,ownGoals,dribbledPast",
        "bigChancesCreated,assists,totalPasses,accuratePassesPercentage",
        "keyPasses,accurateLongBalls,accurateLongBallsPercentage",
        "groundDuelsWon,groundDuelsWonPercentage,aerialDuelsWon,aerialDuelsWonPercentage,totalDuelsWon,totalDuelsWonPercentage",
        "appearances,matchesStarted,minutesPlayed",
    ]
    RENAME_MAP = {
        "totalShots": "Total Shots",
        "goals": "Goals",
        "expectedGoals": "Expected Goals (xG)",
        "goalConversionPercentage": "Goal Conversion %",
        "leftFootGoals": "Left Foot Goals",
        "rightFootGoals": "Right Foot Goals",
        "headedGoals": "Headed Goals",
        "bigChancesMissed": "Big Chances Missed",
        "successfulDribbles": "Successful Dribbles",
        "successfulDribblesPercentage": "Successful Dribbles %",
        "goalsFromInsideTheBox": "Goals From Inside The Box",
        "goalsFromOutsideTheBox": "Goals From Outside The Box",
        "tackles": "Tackles",
        "interceptions": "Interceptions",
        "clearances": "Clearances",
        "cleanSheet": "Clean Sheet",
        "outfielderBlocks": "Outfielder Blocks",
        "penaltyConceded": "Penalty Conceded",
        "errorLeadToGoal": "Error Lead To Goal",
        "errorLeadToShot": "Error Lead To Shot",
        "ownGoals": "Own Goals",
        "dribbledPast": "Dribbled Past",
        "bigChancesCreated": "Big Chances Created",
        "assists": "Assists",
        "totalPasses": "Total Passes",
        "accuratePassesPercentage": "Accurate Passes %",
        "keyPasses": "Key Passes",
        "accurateLongBalls": "Accurate Long Balls",
        "accurateLongBallsPercentage": "Accurate Long Balls %",
        "groundDuelsWon": "Ground Duels Won",
        "groundDuelsWonPercentage": "Ground Duels Won %",
        "aerialDuelsWon": "Aerial Duels Won",
        "aerialDuelsWonPercentage": "Aerial Duels Won %",
        "totalDuelsWon": "Total Duels Won",
        "totalDuelsWonPercentage": "Total Duels Won %",
        "appearances": "Appearances",
        "matchesStarted": "Matches Started",
        "minutesPlayed": "Minutes Played",
    }

    def fetch_page(fields, offset):
        params = {
            "limit": 20,
            "order": "-rating",
            "offset": offset,
            "accumulation": "total",
            "fields": fields,
            "filters": "position.in.G~D~M~F",
        }
        response = requests.get(BASE_URL, headers=HEADERS, params=params, impersonate="chrome110")
        if response.status_code != 200:
            return []
        return response.json().get("results", [])

    def results_to_df(results, fields):
        rows = []
        for item in results:
            row = {
                "Name": item["player"]["name"],
                "Slug": item["player"]["slug"],
                "ID": item["player"]["id"],
                "Team": item["team"]["name"],
            }
            for field in fields.split(","):
                value = item.get(field)
                if field == "expectedGoals" and value is not None:
                    value = round(value, 2)
                row[RENAME_MAP[field]] = value
            rows.append(row)
        return pd.DataFrame(rows)

    # Collect all pages for each field group
    group_dfs = [pd.DataFrame() for _ in FIELD_GROUPS]
    offset = 0

    while True:
        results = fetch_page(FIELD_GROUPS[0], offset)
        if not results:
            break
        for i, fields in enumerate(FIELD_GROUPS):
            page_results = results if i == 0 else fetch_page(fields, offset)
            group_dfs[i] = pd.concat([group_dfs[i], results_to_df(page_results, fields)])
        offset += 20

    group_dfs = [df.reset_index(drop=True) for df in group_dfs]

    # Merge all group DataFrames into one
    merged = group_dfs[0]
    for df in group_dfs[1:]:
        merged = pd.merge(merged, df)

    return merged

In [ ]:
scrape_sofascore_data(17, 76986)

,Name,Slug,ID,Team,Total Shots,Goals,Expected Goals (xG),Goal Conversion %,Left Foot Goals,Right Foot Goals,...,Accurate Long Balls %,Ground Duels Won,Ground Duels Won %,Aerial Duels Won,Aerial Duels Won %,Total Duels Won,Total Duels Won %,Appearances,Matches Started,Minutes Played
0,Tom Edozie,tom-edozie,1445189,Wolverhampton,1,1,0.16,100.00,0,1,...,100.00,2,50.00,0,0.00,2,50.00,1,0,15
1,Bruno Fernandes,bruno-fernandes,288205,Manchester United,67,7,9.19,10.45,0,7,...,59.44,90,50.28,16,36.36,106,47.53,27,27,2347
2,Declan Rice,declan-rice,856714,Arsenal,36,4,2.87,11.11,0,3,...,48.65,77,56.20,44,72.13,121,61.11,30,29,2585
3,Bruno Guimarães,bruno-guimaraes,866469,Newcastle United,32,9,4.82,28.13,1,6,...,60.55,115,51.80,11,34.38,126,49.61,23,22,2018
4,Rodri,rodri,827606,Manchester City,11,1,0.76,9.09,0,0,...,69.03,67,63.21,28,77.78,95,66.90,18,14,1245
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516,Alex Murphy,alex-murphy,1125214,Newcastle United,0,0,NaN,0.00,0,0,...,0.00,0,0.00,0,0.00,0,0.00,1,0,1
517,James Rowswell,james-roswell,1403130,Tottenham Hotspur,0,0,NaN,0.00,0,0,...,0.00,0,0.00,0,0.00,0,0.00,1,0,1
518,Myles Peart-Harris,myles-peart-harris,980649,Brentford,0,0,NaN,0.00,0,0,...,0.00,0,0.00,0,0.00,0,0.00,1,0,1
519,Nehemiah Oriola,nehemiah-oriola,1899611,Brighton & Hove Albion,0,0,NaN,0.00,0,0,...,0.00,0,0.00,0,0.00,0,0.00,1,0,1


In [ ]:
def create_leagues_dict():

  def get_current_season_id(tournament_id):
    url = f"https://api.sofascore.com/api/v1/unique-tournament/{tournament_id}/seasons"
    data = requests.get(url).json()
    return data

  leagues = {
    "country": [
        "England","Italy","Spain","Germany","France",
        "Portugal","Netherlands","Belgium","Turkey","Czech Republic",
        "Greece","Poland","Denmark","Norway","Cyprus",
        "Switzerland","Austria","Scotland","Sweden","Croatia",
        "Israel","Hungary","Serbia","Romania","Ukraine",
        "Slovenia","Azerbaijan","Slovakia","Bulgaria","Russia",
        "Republic of Ireland","Iceland","Armenia","Moldova","Finland",
        "Kosovo","Kazakhstan","Bosnia & Herzegovina","Latvia","Faroe Islands"
    ],

    "league": [
        "Premier League","Serie A","La Liga","Bundesliga","Ligue 1",
        "Liga Portugal","Eredivisie","Pro League","Süper Lig","Czech First League",
        "Super League Greece","Ekstraklasa","Superliga","Eliteserien","Cypriot First Division",
        "Swiss Super League","Austrian Bundesliga","Scottish Premiership","Allsvenskan","HNL",
        "Ligat Ha'AL","Nemzeti Bajnokság I","Serbian SuperLiga","Liga I","Ukrainian Premier League",
        "PrvaLiga","Azerbaijan Premier League","Slovak First League","Parva Liga","Russian Premier League",
        "Premier Division","Besta deild karla","Armenian Premier League","Moldovan Liga","Veikkausliiga",
        "Kosovo Superleague","Kazakh Premier League","Premier League BH","Virslīga","Faroe Islands Premier League"
    ],

    "tournament_id": [
        17, 23, 8, 35, 34,
        238, 37, 38, 52, 172,
        185, 202, 39, 20, 171,
        215, 45, 36, 40, 170,
        266, 187, 210, 152, 218,
        212, 709, 211, 247, 203,
        192, 188, 671, 685, 41,
        9413, 682, 222, 197, 673
    ]
  }
  #20 40 192 188 41 682 197 673
  one_year_season = [20, 40, 192, 188, 41, 682, 197, 673]
  season_ids = [
      get_current_season_id(id).get('seasons')[id in one_year_season].get("id")
      for id in leagues["tournament_id"]
  ]

  leagues["season_id"] = season_ids
  return leagues

In [ ]:
league_dict = create_leagues_dict()
data = pd.DataFrame()


for i in range(len(league_dict['tournament_id'])):
    t_id = league_dict['tournament_id'][i]
    s_id = league_dict['season_id'][i]
    l_name = league_dict['league'][i]
    print(f"Scraping data for {l_name}...")

    try:
        df_league = scrape_sofascore_data(t_id, s_id)
        df_league['League'] = l_name
        data = pd.concat([data, df_league], ignore_index=True)
        time.sleep(1)
    except Exception as e:
        print(f"Error scraping {l_name}: {e}")

print("Scraping completed!")

Scraping data for Premier League...
Scraping data for Serie A...
Scraping data for La Liga...
Scraping data for Bundesliga...
Scraping data for Ligue 1...
Scraping data for Liga Portugal...
Scraping data for Eredivisie...
Scraping data for Pro League...
Scraping data for Süper Lig...
Scraping data for Czech First League...
Scraping data for Super League Greece...
Scraping data for Ekstraklasa...
Scraping data for Superliga...
Scraping data for Eliteserien...
Scraping data for Cypriot First Division...
Scraping data for Swiss Super League...
Scraping data for Austrian Bundesliga...
Scraping data for Scottish Premiership...
Scraping data for Allsvenskan...
Scraping data for HNL...
Scraping data for Ligat Ha'AL...
Scraping data for Nemzeti Bajnokság I...
Scraping data for Serbian SuperLiga...
Scraping data for Liga I...
Scraping data for Ukrainian Premier League...
Scraping data for PrvaLiga...
Scraping data for Azerbaijan Premier League...
Scraping data for Slovak First League...
Scrapin

In [ ]:
data

,Name,Slug,ID,Team,Total Shots,Goals,Expected Goals (xG),Goal Conversion %,Left Foot Goals,Right Foot Goals,...,Ground Duels Won,Ground Duels Won %,Aerial Duels Won,Aerial Duels Won %,Total Duels Won,Total Duels Won %,Appearances,Matches Started,Minutes Played,League
0,Tom Edozie,tom-edozie,1445189,Wolverhampton,1.0,1,0.16,100.00,0,1,...,2,50.00,0.0,0.00,2.0,50.00,1,0,15,Premier League
1,Bruno Fernandes,bruno-fernandes,288205,Manchester United,67.0,7,9.19,10.45,0,7,...,90,50.28,16.0,36.36,106.0,47.53,27,27,2347,Premier League
2,Declan Rice,declan-rice,856714,Arsenal,36.0,4,2.87,11.11,0,3,...,77,56.20,44.0,72.13,121.0,61.11,30,29,2585,Premier League
3,Bruno Guimarães,bruno-guimaraes,866469,Newcastle United,32.0,9,4.82,28.13,1,6,...,115,51.80,11.0,34.38,126.0,49.61,23,22,2018,Premier League
4,Rodri,rodri,827606,Manchester City,11.0,1,0.76,9.09,0,0,...,67,63.21,28.0,77.78,95.0,66.90,18,14,1245,Premier League
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16364,Ingolf Olsen,ingolf-olsen,2294199,B68 Toftir,0.0,0,NaN,0.00,None,None,...,None,NaN,0.0,0.00,0.0,0.00,1,0,2,Faroe Islands Premier League
16365,Ari Joensen,ari-joensen,2056273,FC Suduroy,0.0,0,NaN,0.00,None,None,...,None,NaN,0.0,0.00,1.0,20.00,2,0,9,Faroe Islands Premier League
16366,Markus Hellisdal,markus-hellisdal,1029302,B36 Tórshavn,0.0,0,NaN,0.00,None,None,...,None,NaN,0.0,0.00,0.0,0.00,2,0,9,Faroe Islands Premier League
16367,Jákup Hummeland,jakup-hummeland,1067422,EB/Streymur,0.0,0,NaN,0.00,None,None,...,None,NaN,0.0,0.00,1.0,12.50,2,0,12,Faroe Islands Premier League


In [ ]:
data.to_csv('sofascore_stats.csv')

In [ ]:
from datetime import datetime

mockdata = {
    'slug': ['declan-rice','bruno-fernandes'],
    'id': [856714, 288205] # ID thực tế trên Sofascore
}
df = pd.DataFrame(mockdata)

def get_player_details(player_id):
    url = f"https://api.sofascore.com/api/v1/player/{player_id}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
        "Referer": "https://www.sofascore.com/"
    }
    response = requests.get(url, headers=headers, impersonate="chrome110")
    if response.status_code == 200:
        time.sleep(0.2)
        data = response.json()
        player = data.get('player', {})

        nationality = player.get('country', {}).get('name')
        dob_str = player.get('dateOfBirth')
        pos = player.get('positionsDetailed')
        prefered_foot = player.get('preferredFoot')
        contract_ts = player.get('contractUntilTimestamp')
        value = player.get('proposedMarketValueRaw', {}).get('value')
        value_currency = player.get('proposedMarketValueRaw', {}).get('currency')

        age = "N/A"
        if dob_str:
            dob = datetime.fromisoformat(dob_str.replace('Z', '+00:00'))
            today = datetime.now(dob.tzinfo)
            age = today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day))

        contract_date = datetime.fromtimestamp(contract_ts).strftime('%Y-%m-%d') if contract_ts else "N/A"

        player_dict = {
            "Nationality": nationality,
            "Age": age,
            "Position": pos,
            "Preferred Foot": prefered_foot,
            "Contract Until": contract_date,
            "Value": value,
            "Currency": value_currency
        }

        return player_dict

# Example usage:
rice_details = get_player_details(856714)
display(rice_details)

{'Nationality': 'England',
 'Age': 27,
 'Position': ['MC', 'DM'],
 'Preferred Foot': 'Right',
 'Contract Until': '2028-06-30',
 'Value': 114000000,
 'Currency': 'EUR'}

In [ ]:
from tqdm import tqdm

# Kích hoạt thanh tiến trình cho pandas apply
tqdm.pandas(desc="Đang lấy thông tin cầu thủ")

# 1. Định nghĩa một hàm bọc (wrapper) để xử lý trường hợp API lỗi (hàm của bạn đang trả về None nếu status_code != 200)
def safe_get_details(player_id):
    result = get_player_details(player_id)
    # Nếu kết quả là None (lỗi API), trả về dict rỗng để không bị lỗi khi chuyển thành Series
    if result is None:
        return {}
    return result

# 2. Áp dụng hàm lên cột 'ID' và chuyển dictionary thành các cột tương ứng
new_columns = data['ID'].progress_apply(lambda x: pd.Series(safe_get_details(x)))

# 3. Nối (concat) các cột mới này vào dataframe 'data' ban đầu
data = pd.concat([data, new_columns], axis=1)

# Kiểm tra kết quả
print(data.head())

Đang lấy thông tin cầu thủ: 100%|██████████| 16369/16369 [1:23:08<00:00,  3.28it/s]

              Name             Slug       ID               Team  Total Shots  \
0       Tom Edozie       tom-edozie  1445189      Wolverhampton          1.0   
1  Bruno Fernandes  bruno-fernandes   288205  Manchester United         67.0   
2      Declan Rice      declan-rice   856714            Arsenal         36.0   
3  Bruno Guimarães  bruno-guimaraes   866469   Newcastle United         32.0   
4            Rodri            rodri   827606    Manchester City         11.0   

   Goals  Expected Goals (xG)  Goal Conversion % Left Foot Goals  \
0      1                 0.16             100.00               0   
1      7                 9.19              10.45               0   
2      4                 2.87              11.11               0   
3      9                 4.82              28.13               1   
4      1                 0.76               9.09               0   

  Right Foot Goals  ... Matches Started Minutes Played          League  \
0                1  ...             